# Anchor Calibration Simulation
## Demonstrating Value on Anomalous vs Normal Days

This notebook simulates price shifts (flash sales, price increases) to prove that:
- On **normal days**: calibration is neutral (no harm)
- On **anomalous days**: calibration reduces error by 91-95%

In [1]:
import os
import sys
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, os.path.dirname(os.getcwd()))

from src.data_loader import load_data, split_test_anchors
from src.feature_engineering import build_features
from src.model_global import (
    train_global_model, predict_global,
    create_time_based_split, simulate_anchor_validation
)
from src.anchor_calibration import apply_global_bias_correction
from src.evaluate import compute_metrics

SEED = 42
np.random.seed(SEED)

## Setup: Load Data & Train Model

In [2]:
train_df, test_df = load_data()
train_split, val_split = create_time_based_split(train_df, n_val_days=3)
val_anchors, val_targets = simulate_anchor_validation(val_split, n_anchors=100)

train_feat = build_features(train_split, train_split, is_training=True)
val_targets_feat = build_features(val_targets, train_split, is_training=False)
val_anchors_feat = build_features(val_anchors, train_split, is_training=False)

model, _ = train_global_model(train_feat, val_targets_feat, save_model=False)

# Raw predictions (fixed - model doesn't change)
raw_preds = predict_global(model, val_targets_feat)
anchor_preds = predict_global(model, val_anchors_feat)

Loading training data...


Loading test data...


Training data: (306226, 26)
Test data: (25900, 26)
Training dates: 2025-01-01 to 2025-03-19 (56 days)
Validation dates: 2025-03-20 to 2025-03-22 (3 days)
Train split: 285307 rows
Val split: 20919 rows


Simulated anchors: 300 rows
Simulated targets: 20619 rows
Building features...


  ✓ Temporal features
  ✓ Discount features
  ✓ Price range features
  ✓ Shop features


...

[1000]	valid_0's l1: 174565


[1100]	valid_0's l1: 173580


[1200]	valid_0's l1: 172853


Early stopping, best iteration is:
[1197]	valid_0's l1: 172839


Best iteration: 1197

Top 15 features:
            feature  importance
    hist_price_mean       20998
       date_ordinal       17703
 shop_quality_score       15188
     item_price_min       14801
     hist_price_std       14703
        shop_rating       14416
       day_of_month       13070
 shop_response_rate       12611
            modelId       12289
shop_follower_count       11393
     item_price_max       11001
     hist_price_min       10955
               hour        9365
         hist_count        9275
priceBeforeDiscount        9035


## Scenario 1: Normal Day (No Price Shift)

In [3]:
actual = val_targets['price'].values
anchor_actual = val_anchors['price'].values

metrics_raw = compute_metrics(actual, raw_preds)

calibrated = apply_global_bias_correction(
    raw_preds, anchor_preds, anchor_actual, method='multiplicative'
)
metrics_cal = compute_metrics(actual, calibrated)

print(f"Without calibration: MAPE = {metrics_raw['mape']:.2f}%")
print(f"With calibration:    MAPE = {metrics_cal['mape']:.2f}%")
print(f"→ Calibration is NEUTRAL on normal days")

  Global multiplicative correction: 1.0000
Without calibration: MAPE = 0.85%
With calibration:    MAPE = 0.85%
→ Calibration is NEUTRAL on normal days


## Scenario 2: Flash Sale (-15% Platform-Wide)

In [4]:
shift = 0.85  # 15% drop
shifted_actual = actual * shift
shifted_anchor_actual = anchor_actual * shift

# Without calibration
m_no = compute_metrics(shifted_actual, raw_preds)

# With calibration
cal = apply_global_bias_correction(
    raw_preds, anchor_preds, shifted_anchor_actual, method='multiplicative'
)
m_yes = compute_metrics(shifted_actual, cal)

improvement = (1 - m_yes['mape'] / m_no['mape']) * 100

print(f"Without calibration: MAPE = {m_no['mape']:.2f}%")
print(f"With calibration:    MAPE = {m_yes['mape']:.2f}%")
print(f"Error reduction:     {improvement:.1f}%")
print(f"→ Calibration SAVES predictions during flash sales!")

  Global multiplicative correction: 0.8500
Without calibration: MAPE = 18.27%
With calibration:    MAPE = 0.85%
Error reduction:     95.4%
→ Calibration SAVES predictions during flash sales!


## Scenario 3: Price Increase (+10%)

In [5]:
shift = 1.10  # 10% increase
shifted_actual = actual * shift
shifted_anchor_actual = anchor_actual * shift

m_no = compute_metrics(shifted_actual, raw_preds)
cal = apply_global_bias_correction(
    raw_preds, anchor_preds, shifted_anchor_actual, method='multiplicative'
)
m_yes = compute_metrics(shifted_actual, cal)
improvement = (1 - m_yes['mape'] / m_no['mape']) * 100

print(f"Without calibration: MAPE = {m_no['mape']:.2f}%")
print(f"With calibration:    MAPE = {m_yes['mape']:.2f}%")
print(f"Error reduction:     {improvement:.1f}%")

  Global multiplicative correction: 1.1000
Without calibration: MAPE = 9.76%
With calibration:    MAPE = 0.85%
Error reduction:     91.3%


## Summary

| Scenario | Without Calibration | With Calibration | Error Reduction |
|----------|--------------------|--------------------|------------------|
| Normal day | MAPE 0.85% | MAPE 0.85% | 0% (neutral) |
| Flash sale -15% | MAPE 18.27% | MAPE 0.85% | **95.4%** |
| Price increase +10% | MAPE 9.76% | MAPE 0.85% | **91.3%** |

**Key Insight**: Anchor calibration is a safety net. It doesn't degrade normal predictions, but dramatically corrects systematic price shifts when they occur.